<a href="https://colab.research.google.com/github/marcohuertas/hugging_face_projects/blob/main/token_classification_value_units_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Token Classification (Version-01)

This notebook contains the code to setup and calibrate a LLM in a token classification task. The tokens to be classified represent the value and unit of a measurement in a sentence.

In this version only one unit has been processed: $^∘$C

- Dataset: "bowenxian/BioProBench"
- Model checkpoint: "bert-base-cased"

## Install packages needed

In [1]:
!pip install seqeval
!pip install evaluate

## Prepare data

In [2]:
import re
from functools import reduce
import numpy as np
from huggingface_hub import notebook_login
from datasets import load_dataset
from datasets import Features, ClassLabel, List
from unicodedata import normalize
from transformers import AutoTokenizer
from transformers import DataCollatorForTokenClassification
from transformers import AutoModelForTokenClassification
from transformers import TrainingArguments
from transformers import Trainer
import evaluate

In [3]:
notebook_login()

### Prepare NER data
Use the data from dataset bowenxian/BioProBench to create a token classifiction dataset, where the entities are the values and scientific units found in a sentence.
Labels:
- values: B-VALUE, I-VALUE
- units: B-UNIT, I-UNIT



In [4]:
ds_base = load_dataset("bowenxian/BioProBench", data_files="PQA.json")

def fill_with_answer(example):
  return re.sub(r"_+", example['answer'], example['question'])

ds_base = (
    ds_base['train']
    .filter(lambda u: u['type']=='parameter')
    .filter(lambda u: u['id']!='PQA-PAR-B008888')
    .map(
        lambda u: {'sentence': [re.sub(r"__+", a, q) for a, q in zip(u['answer'], u['question'])]},
        batched=True)
    )

Limit analysis to one unit: $^\circ$C

In [5]:
in_cond = lambda u, regexp: normalize('NFD', regexp) in normalize('NFD', u['sentence'])

# Filter for degree Celsius
regexp = "\u00B0C"
ds = ds_base.filter(lambda u: in_cond(u, regexp))

### Pre-processing
Prepare the data by spitting the entries in 'sentence' into a list of words.

In [6]:
# Split by white space and parenthesis, then split the '.' from any word, but not from numbers.

def split_word_period(sample):
  regexp = r'([A-Za-z]+[.])'
  o = re.search(regexp, sample)
  if o:
    output = [u for u in re.split('([.])', sample) if u != '']
  else:
    output = [sample]

  return output

def apply_split_word_period(examples):
  output = []
  for sentence in examples['sentence']:
    sentence_split = [u for u in re.split(r'([\s()])', sentence) if u not in [' ','']]
    # output.append(reduce(lambda p,q: p+q, [split_word_period(u) for u in sentence.split()]))
    output.append(reduce(lambda p,q: p+q, [split_word_period(u) for u in sentence_split]))

  return {'words': output}

ds = ds.map(apply_split_word_period, batched=True, batch_size=100)

In [7]:
def split_comma(examples):
  regexp = r'([0-9]+,[0-9]+)'

  output = []
  for words in examples['words']:
    new_words_split = []

    for word in words:
      # check that it has a comma
      # and that is not a number (e.g 3,500)
      has_comma = "," in word
      check = re.search(regexp, word)
      if has_comma and check is None:
        word_split = [u for u in re.split("(,)", word) if u != '']
        new_words_split.append(word_split)
      else:
        new_words_split.append([word])

    output.append(reduce(lambda p,q: p+q, new_words_split))

  return {'words': output}

ds = ds.map(split_comma, batched=True, batch_size=100)


### Pre-label data

The output of the labeling should be something like this:

[50$^∘$C, and, 12.345, $^∘$C]

['B-COMBINED', '0', 'B-VALUE', 'B-UNIT']

This will be split by the tokenizer as

['[CLS]', '50', '##°', '##C', 'and', '12', '.', '345', '°C', '[SEP]']

In [8]:
def is_numeric(string: str) -> bool:
    # Try to convert the string to a float
    # If the conversion is successful, return True
    try:
        float(string)
        return True
    # If a ValueError is thrown, it means the conversion was not successful
    # This happens when the string contains non-numeric characters
    except ValueError:
        return False

def contains_a_number(sample):
  return bool(re.search(r'\d', sample))

def pre_ner_label(sample):

  names = ['O', 'B-VALUE', 'I-VALUE', 'B-UNIT', 'I-UNIT', 'B-COMBINED', 'I-COMBINED']
  label2id = {k:idx for idx,k in enumerate(names)}

  degreeCelcius = normalize('NFD', "\u00B0C")
  ner_labels = ['O']*len(sample)
  ner_tags = [0]*len(sample)

  for idx, word in enumerate(sample):
    contains_unit = re.search(degreeCelcius, normalize('NFD', word))
    if contains_unit is not None:
      pos_start, pos_end = contains_unit.span()
      # Contains the unit
      if pos_start == 0:
        label = 'B-UNIT'
        ner_labels[idx] = label
        ner_tags[idx] = label2id[label]

        # if is_numeric(sample[idx-1]):
        if contains_a_number(sample[idx-1]):
          label = 'B-VALUE'
          ner_labels[idx-1] = label
          ner_tags[idx-1] = label2id[label]
      else:
        label = 'B-COMBINED'
        ner_labels[idx] = label
        ner_tags[idx] = label2id[label]

  return ner_labels, ner_tags

def apply_pre_ner_label(examples):
  samples = examples['words']
  output_labels = []
  output_tags = []
  for sample in samples:
    ner_labels, ner_tags = pre_ner_label(sample)
    output_labels.append(ner_labels)
    output_tags.append(ner_tags)

  return {
      'pre_ner_labels': output_labels,
      'pre_ner_tags': output_tags,
      }

ds = ds.map(apply_pre_ner_label, batched=True, batch_size=100)

## Prepare Tokenized Dataset for Training

In [9]:
# NOTE: This function was provided by HF in the token classification example worked out.
# For detailes see: https://huggingface.co/learn/llm-course/chapter7/2

# To adjust the NER tags use this function
# Here labels==ner_tags
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            # Start of a new word!
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            # Special token
            new_labels.append(-100)
        else:
            # Same word as previous token
            label = labels[word_id]
            # If the label is B-XXX we change it to I-XXX
            # MH: This assumes a specific way of arranging the ner-labels
            # see output of ds['train'].features['ner_tags']
            if label % 2 == 1:
                label += 1
            new_labels.append(label)

    return new_labels

Convert the label B-COMBINED into B-VALUE, I-VALUE and B-UNIT, I-UNIT.
This label occurs when the value and the unit don't have a space in between.
In general, there should be a space, but I didn't want to restrict my analysis to only those cases.

1. Take a 'words' list and tokenize it.
2. Align the pre-ner-tags based on the tokenization result
3. Convert B-COMBINED, I-COMBINED by checking where he unit is located and relabel these tokens as B-UNIT, I-UNIT and B-VALUE, I-VALUE
4. Call the output 'labels'

In [10]:
def token_in_unit(token, unit):
  word_part = token
  if '##' in token:
    word_part = token.split('##')[-1]

  return word_part in unit

def generate_ner_labels(tokens, labels):
  unit = "\u00B0C"
  names = ['O', 'B-VALUE', 'I-VALUE', 'B-UNIT', 'I-UNIT']
  label2id = {k:idx for idx,k in enumerate(names)}

  new_labels = labels.copy()
  ner_tag = 'O'
  # for idx, (token, label) in enumerate(zip(inputs.tokens(), labels)):
  for idx, (token, label) in enumerate(zip(tokens, labels)):
    if label>=5:
      if token_in_unit(token, unit):
        # If token is part of unit
        if ner_tag in ['B-VALUE', 'I-VALUE']:
          ner_tag = 'B-UNIT'
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id
        elif ner_tag == 'B-UNIT':
          ner_tag = 'I-UNIT'
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id
        elif ner_tag == 'I-UNIT':
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id

      else:
        # If token not part of unit
        if ner_tag == 'O':
          ner_tag = 'B-VALUE'
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id
        elif ner_tag == 'B-VALUE':
          ner_tag = 'I-VALUE'
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id
        elif ner_tag == 'I-VALUE':
          ner_label_id = label2id[ner_tag]
          new_labels[idx] = ner_label_id
    else:
      ner_tag = 'O'

  return new_labels

def tokenize_and_generate_labels(examples):
  tokenized_inputs = tokenizer(
      examples['words'],
      is_split_into_words=True,
      truncation=True
      )
  # Get the preliminary ner tags
  pre_ner_tags = examples['pre_ner_tags']
  # Align labels
  old_labels = [align_labels_with_tokens(tags, tokenized_inputs.word_ids(idx)) for idx, tags in enumerate(pre_ner_tags)]
  # Generate new labels
  new_labels = [generate_ner_labels(tokenized_inputs.tokens(idx), labels) for idx, labels in enumerate(old_labels)]

  tokenized_inputs["temp_labels"] = new_labels

  return tokenized_inputs

In [11]:
model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [51]:
tokenized_dataset = ds.map(
    tokenize_and_generate_labels,
    batched=True,
    remove_columns=ds.column_names
    )

tokenized_dataset = tokenized_dataset.add_column(
    name='labels',
    column=tokenized_dataset['temp_labels'],
    feature=List(ClassLabel(names=['O', 'B-VALUE', 'I-VALUE', 'B-UNIT', 'I-UNIT']))
    ).remove_columns(column_names='temp_labels')


Map:   0%|          | 0/6565 [00:00<?, ? examples/s]

There were some cases where the unit of $^\circ$C appeared in unseen formats, like $^\circ$C/s that the code did not handle properly.

Filter data to remove those cases.

In [53]:
tokenized_dataset = tokenized_dataset.filter(lambda u: max(u['labels']) <= 4)

Filter:   0%|          | 0/6565 [00:00<?, ? examples/s]

### Prepare a train, validate, test split

In [15]:
def split_ds_train_test_val(ds):
  seed = 56
  # Create a train, test split
  ds_train_test = ds.train_test_split(train_size=0.9, seed=seed)

  # Use the previous 'train' to create a new split
  ds_final = ds_train_test['train'].train_test_split(train_size=0.8, seed=seed)

  # Replace the 'test' split with one named 'validation'
  ds_final['validation'] = ds_final.pop('test')

  # Add the 'test' dataset from the first split
  ds_final['test'] = ds_train_test['test']

  return ds_final

In [16]:
tokenized_dataset_final = split_ds_train_test_val(tokenized_dataset)

## Train model

Follow steps from the token classification tutorial

In [17]:
metric = evaluate.load("seqeval")

# Function to compute metrics
# label_names = ds_samples_final['train'].features['ner_tags'].feature.names
label_names = tokenized_dataset_final['train'].features['labels'].feature.names

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [18]:
# These are required for the token classification task
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

args = TrainingArguments(
    "/content/bert-finetuned-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset_final["train"],
    eval_dataset=tokenized_dataset_final["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.033453,0.004508,0.982967,0.991600,0.987265,0.998241
2,0.003956,0.004253,0.989341,0.992363,0.990850,0.998426
3,0.001831,0.003546,0.990861,0.993509,0.992183,0.998704


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1764, training_loss=0.011425149751628607, metrics={'train_runtime': 251.5433, 'train_samples_per_second': 56.03, 'train_steps_per_second': 7.013, 'total_flos': 384293234683020.0, 'train_loss': 0.011425149751628607, 'epoch': 3.0})

## Test some examples

In [26]:
from transformers import pipeline

# Replace this with your own checkpoint
# model_checkpoint = "huggingface-course/bert-finetuned-ner"
model_checkpoint = "/content/bert-finetuned-ner/checkpoint-1764"
token_classifier = pipeline(
    "token-classification", model=model_checkpoint, aggregation_strategy="simple"
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [45]:
sample_str = ds[-5]['sentence']
print(sample_str)
output = token_classifier(sample_str)
print(output)


Both solutions should be warmed to 37 °C before perfusion.
[{'entity_group': 'VALUE', 'score': np.float32(0.9998129), 'word': '37', 'start': 35, 'end': 37}, {'entity_group': 'UNIT', 'score': np.float32(0.9998896), 'word': '°C', 'start': 38, 'end': 40}]
